# Лабораторная работа № 3


## Подготовка

In [5]:
import os, json, shutil, logging, random, sys, importlib
from pathlib import Path
from typing import Iterator
from logging.handlers import RotatingFileHandler

random.seed(42)

## Задание 1. Функция и процедура

In [6]:
def log_error(path: str, reason: str) -> None:
    """Логирует путь к файлу и причину ошибки; полезное значение не возвращает."""
    logging.warning("Ошибка в файле %s: %s", path, reason)

def parse_int_safe(value: str) -> int | None:
    """Преобразует строку в целое число или возвращает None при ValueError."""
    try:
        return int(value)
    except ValueError:
        return None

logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")
log_error("data.txt", "не удалось прочитать")
print(parse_int_safe("123"), parse_int_safe("abc"))
# Процедура выполняет действие; функция вычисляет и возвращает результат.

123 None


## Задание 2. Типы исключений и MRO

In [7]:
tests = [
    ("1/0", lambda: 1/0),
    ('int("abc")', lambda: int("abc")),
    ("[1, 2][5]", lambda: [1, 2][5]),
    ('{"a": 1}["b"]', lambda: {"a": 1}["b"]),
    ('"5" + 5', lambda: "5" + 5),
    ('open("no_such_file.txt")', lambda: open("no_such_file.txt", encoding="utf-8")),
]
for expression, operation in tests:
    try:
        value = operation()
        if hasattr(value, "close"): value.close()
        print(expression, "->", value)
    except (ArithmeticError, LookupError, OSError, ValueError, TypeError) as exc:
        print(expression, "->", type(exc).__name__)
        print("MRO:", " -> ".join(c.__name__ for c in type(exc).__mro__))
# Например: ZeroDivisionError наследуется от ArithmeticError; KeyError и IndexError — LookupError;
# FileNotFoundError — OSError; ValueError и TypeError — непосредственно от Exception.

1/0 -> ZeroDivisionError
MRO: ZeroDivisionError -> ArithmeticError -> Exception -> BaseException -> object
int("abc") -> ValueError
MRO: ValueError -> Exception -> BaseException -> object
[1, 2][5] -> IndexError
MRO: IndexError -> LookupError -> Exception -> BaseException -> object
{"a": 1}["b"] -> KeyError
MRO: KeyError -> LookupError -> Exception -> BaseException -> object
"5" + 5 -> TypeError
MRO: TypeError -> Exception -> BaseException -> object
open("no_such_file.txt") -> FileNotFoundError
MRO: FileNotFoundError -> OSError -> Exception -> BaseException -> object


## Задание 3. try / except / else / finally

In [8]:
def safe_divide(a, b) -> float | None:
    """Делит a на b; при делении на ноль записывает ошибку и возвращает None."""
    try:
        result = a / b
    except ZeroDivisionError as exc:
        logging.error("Деление на ноль: %s", exc)
        return None
    else:
        print("Деление выполнено успешно")
        return result
    finally:
        print("Операция завершена")

for pair in [(10, 2), (10, 0), (10, "x")]:
    try: print(pair, "->", safe_divide(*pair))
    except TypeError as exc: print(pair, "-> TypeError:", exc)
# finally выполняется и перед выходом через return.

ERROR: Деление на ноль: division by zero


Деление выполнено успешно
Операция завершена
(10, 2) -> 5.0
Операция завершена
(10, 0) -> None
Операция завершена
(10, 'x') -> TypeError: unsupported operand type(s) for /: 'int' and 'str'


## Задание 4. Иерархия собственных исключений

In [9]:
class IngestError(Exception):
    """Базовая ошибка импорта."""

class ConfigError(IngestError):
    """Неверная конфигурация."""

class DocumentError(IngestError):
    """Ошибка одного документа."""
    def __init__(self, path, reason):
        self.path, self.reason = str(path), reason
        super().__init__(f"{self.path}: {self.reason}")

for err in [ConfigError("неверные настройки"), DocumentError("a.json", "нет поля id")]:
    try: raise err
    except IngestError as exc: print(type(exc).__name__, exc)
# except IngestError перехватывает оба исключения-наследника.

ConfigError неверные настройки
DocumentError a.json: нет поля id


## Задание 5. LBYL и EAFP

In [10]:
def read_lbyl(path: str | Path) -> str | None:
    """Сначала проверяет наличие файла, затем читает его."""
    path = Path(path)
    if path.exists() and path.is_file():
        return path.read_text(encoding="utf-8")
    return None

def read_eafp(path: str | Path) -> str | None:
    """Пытается открыть файл и обрабатывает отсутствие."""
    try: return Path(path).read_text(encoding="utf-8")
    except FileNotFoundError: return None

demo = Path("lbyl_eafp_demo.txt")
demo.write_text("Пример", encoding="utf-8")
print(read_lbyl(demo), read_eafp(demo), read_eafp("missing.txt"))
# EAFP надёжнее в гонке состояний: файл может исчезнуть после exists().

Пример Пример None


## Задание 6. Режимы файлов

In [11]:
p = Path("test.txt")
with p.open("w", encoding="utf-8") as f: f.write("Первая\nВторая\nТретья\n")
print(p.read_text(encoding="utf-8"))
with p.open("r", encoding="utf-8") as f:
    for line in f: print(line.rstrip())
with p.open("a", encoding="utf-8") as f: f.write("Четвёртая\n")
print("Число строк:", len(p.read_text(encoding="utf-8").splitlines()))
# Повторное открытие в режиме w усечёт файл и удалит старое содержимое.

Первая
Вторая
Третья

Первая
Вторая
Третья
Число строк: 4


## Задание 7. JSON

In [12]:
data = {"название": "Обработка документов", "список": [1, 2, 3], "текст": "Привет"}
p = Path("sample.json")
with p.open("w", encoding="utf-8") as f: json.dump(data, f, ensure_ascii=False, indent=2)
with p.open("r", encoding="utf-8") as f: loaded = json.load(f)
print("Совпадают:", data == loaded, loaded)
try: json.loads('{"broken": }')
except json.JSONDecodeError as exc: print("Пойман JSONDecodeError:", exc)

Совпадают: True {'название': 'Обработка документов', 'список': [1, 2, 3], 'текст': 'Привет'}
Пойман JSONDecodeError: Expecting value: line 1 column 12 (char 11)


## Задание 8. pathlib

In [13]:
base = Path("pathlib_demo")
raw, processed = base / "data" / "raw", base / "data" / "processed"
raw.mkdir(parents=True, exist_ok=True)
processed.mkdir(parents=True, exist_ok=True)
(raw / "a.json").write_text('{"id": 1}', encoding="utf-8")
(raw / "b.json").write_text('{"id": 2}', encoding="utf-8")
for file in sorted(base.rglob("*.json")):
    print(file.name, file.stat().st_size, file.resolve())

a.json 9 D:\maestria\I semestre\python-labs-gutierrez\pathlib_demo\data\raw\a.json
b.json 9 D:\maestria\I semestre\python-labs-gutierrez\pathlib_demo\data\raw\b.json


## Задание 9. Модуль jsonutils.py

In [14]:
module_text = '''
import json
from pathlib import Path

class DocumentError(Exception):
    """Ошибка чтения JSON."""

def read_json(path: str | Path) -> dict:
    """Читает JSON-словарь или поднимает DocumentError."""
    try:
        with Path(path).open(encoding="utf-8") as f:
            data = json.load(f)
    except (OSError, json.JSONDecodeError) as exc:
        raise DocumentError(f"{path}: {exc}") from exc
    if not isinstance(data, dict):
        raise DocumentError(f"{path}: JSON должен быть объектом")
    return data

def write_json(data: dict, path: str | Path) -> None:
    """Записывает словарь в JSON, сохраняя кириллицу."""
    with Path(path).open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

if __name__ == "__main__":
    write_json({"id": 1, "name": "Пример"}, "jsonutils_demo.json")
    print(read_json("jsonutils_demo.json"))
'''
Path("jsonutils.py").write_text(module_text.strip() + "\n", encoding="utf-8")
importlib.invalidate_caches()
import jsonutils
jsonutils.write_json({"id": 1, "name": "Тест"}, "jsonutils_demo.json")
print(jsonutils.read_json("jsonutils_demo.json"))

{'id': 1, 'name': 'Тест'}


## Задание 10. Логирование с уровнями и ротацией

In [15]:
def setup_task10_logging():
    """Консольный INFO и файловый DEBUG с ротацией app.log."""
    logger = logging.getLogger("task10")
    logger.setLevel(logging.DEBUG); logger.handlers.clear(); logger.propagate = False
    console = logging.StreamHandler(); console.setLevel(logging.INFO)
    file_handler = RotatingFileHandler("app.log", maxBytes=100_000, backupCount=3, encoding="utf-8")
    file_handler.setLevel(logging.DEBUG)
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(name)s %(message)s")
    console.setFormatter(fmt); file_handler.setFormatter(fmt)
    logger.addHandler(console); logger.addHandler(file_handler)
    return logger

log10 = setup_task10_logging()
for level, message in [(10,"DEBUG"),(20,"INFO"),(30,"WARNING"),(40,"ERROR"),(50,"CRITICAL")]:
    log10.log(level, message)
print("DEBUG отсутствует в консоли, потому что консольный уровень — INFO.")
print(Path("app.log").read_text(encoding="utf-8"))

2026-10-09 13:11:38,418 INFO task10 INFO
2026-10-09 13:11:38,419 WARNING task10 WARNING
2026-10-09 13:11:38,420 ERROR task10 ERROR
2026-10-09 13:11:38,420 CRITICAL task10 CRITICAL


DEBUG отсутствует в консоли, потому что консольный уровень — INFO.
2026-10-09 13:11:38,416 DEBUG task10 DEBUG
2026-10-09 13:11:38,418 INFO task10 INFO
2026-10-09 13:11:38,419 WARNING task10 WARNING
2026-10-09 13:11:38,420 ERROR task10 ERROR
2026-10-09 13:11:38,420 CRITICAL task10 CRITICAL



## Общая инициализация данных для заданий 11–20

In [16]:
shutil.rmtree("inbox", ignore_errors=True)
Path("inbox/good").mkdir(parents=True)
Path("inbox/nested").mkdir(parents=True)
Path("inbox/good/a.json").write_text('{"id": 1, "name": "Alice", "age": 30}', encoding="utf-8")
Path("inbox/good/b.json").write_text('{"id": 2, "name": "Bob", "age": 25}', encoding="utf-8")
Path("inbox/nested/c.json").write_text('{"id": 3, "name": "Carol", "age": 28}', encoding="utf-8")
Path("inbox/broken.json").write_text('{"id": 4, "name": }', encoding="utf-8")
Path("inbox/no_keys.json").write_text('{"foo": "bar"}', encoding="utf-8")
Path("inbox/not_object.json").write_text('[1, 2, 3]', encoding="utf-8")
print("JSON-файлов:", len(list(Path("inbox").rglob("*.json"))))

JSON-файлов: 6


## Задание 11. Настройки из окружения

In [17]:
class Settings:
    """Хранит источник, путь отчёта и обязательные поля."""
    def __init__(self, source_dir=Path("inbox"), output_path=Path("catalog.json"),
                 required_keys=("id", "name")):
        self.source_dir, self.output_path = Path(source_dir), Path(output_path)
        self.required_keys = tuple(required_keys)

    @classmethod
    def from_env(cls):
        """Читает переменные окружения и проверяет настройки."""
        src = Path(os.environ.get("SOURCE_DIR", "inbox"))
        out = Path(os.environ.get("OUTPUT_PATH", "catalog.json"))
        keys = tuple(k.strip() for k in os.environ.get("REQUIRED_KEYS", "id,name").split(",") if k.strip())
        if not src.is_dir(): raise ConfigError(f"Каталог-источник не существует: {src}")
        if not keys: raise ConfigError("REQUIRED_KEYS не должен быть пустым")
        return cls(src, out, keys)

settings = Settings.from_env()
print(settings.source_dir, settings.output_path, settings.required_keys)

inbox catalog.json ('id', 'name')


## Задание 12. Рекурсивный обход

In [18]:
class Ingester:
    """Пакетно импортирует JSON-документы с изоляцией ошибок файлов."""
    def __init__(self, settings: Settings, logger=None):
        self.settings = settings
        self.logger = logger or logging.getLogger("ingester")

    def iter_files(self) -> Iterator[Path]:
        """Возвращает отсортированный итератор JSON-файлов из подкаталогов."""
        return iter(sorted(self.settings.source_dir.rglob("*.json")))

print([str(p) for p in Ingester(settings).iter_files()])
# rglob компактно выполняет рекурсивный поиск по шаблону; os.walk полезен для более тонкого контроля обхода.

['inbox\\broken.json', 'inbox\\good\\a.json', 'inbox\\good\\b.json', 'inbox\\nested\\c.json', 'inbox\\no_keys.json', 'inbox\\not_object.json']


## Задание 13. Чтение и валидация документа

In [19]:
def load_document(self, path: Path) -> dict:
    """Читает JSON и проверяет словарь и обязательные поля."""
    try:
        with path.open("r", encoding="utf-8") as f: data = json.load(f)
    except (OSError, json.JSONDecodeError) as exc:
        raise DocumentError(path, str(exc)) from exc
    if not isinstance(data, dict):
        raise DocumentError(path, "JSON должен быть объектом (словарём)")
    missing = [key for key in self.settings.required_keys if key not in data]
    if missing: raise DocumentError(path, "отсутствуют поля: " + ", ".join(missing))
    return data

Ingester.load_document = load_document
for path in Ingester(settings).iter_files():
    try: print(path, "->", Ingester(settings).load_document(path))
    except DocumentError as exc: print("Пропущен:", exc)

Пропущен: inbox\broken.json: Expecting value: line 1 column 19 (char 18)
inbox\good\a.json -> {'id': 1, 'name': 'Alice', 'age': 30}
inbox\good\b.json -> {'id': 2, 'name': 'Bob', 'age': 25}
inbox\nested\c.json -> {'id': 3, 'name': 'Carol', 'age': 28}
Пропущен: inbox\no_keys.json: отсутствуют поля: id, name
Пропущен: inbox\not_object.json: JSON должен быть объектом (словарём)


## Задание 14. Конвейер с изоляцией ошибок

In [20]:
def run(self) -> dict:
    """Импортирует все файлы, продолжая работу после ошибки отдельного документа."""
    files = list(self.iter_files()); records, errors = [], []
    for path in files:
        try: records.append(self.load_document(path))
        except DocumentError as exc:
            self.logger.warning("Документ пропущен: %s", exc)
            errors.append({"path": str(path), "reason": exc.reason})
        except Exception as exc:
            self.logger.exception("Неожиданная ошибка в %s", path)
            errors.append({"path": str(path), "reason": str(exc)})
    return {"source": str(self.settings.source_dir), "total_files": len(files),
            "imported": len(records), "skipped": len(errors), "errors": errors, "records": records}

Ingester.run = run
print(json.dumps(Ingester(settings).run(), ensure_ascii=False, indent=2))

{
  "source": "inbox",
  "total_files": 6,
  "imported": 3,
  "skipped": 3,
  "errors": [
    {
      "path": "inbox\\broken.json",
      "reason": "Expecting value: line 1 column 19 (char 18)"
    },
    {
      "path": "inbox\\no_keys.json",
      "reason": "отсутствуют поля: id, name"
    },
    {
      "path": "inbox\\not_object.json",
      "reason": "JSON должен быть объектом (словарём)"
    }
  ],
  "records": [
    {
      "id": 1,
      "name": "Alice",
      "age": 30
    },
    {
      "id": 2,
      "name": "Bob",
      "age": 25
    },
    {
      "id": 3,
      "name": "Carol",
      "age": 28
    }
  ]
}


## Задание 15. Сохранение отчёта

In [21]:
def _save(self, report: dict) -> None:
    """Сохраняет JSON-отчёт; ошибку записи оборачивает в ConfigError."""
    try:
        self.settings.output_path.parent.mkdir(parents=True, exist_ok=True)
        with self.settings.output_path.open("w", encoding="utf-8") as f:
            json.dump(report, f, ensure_ascii=False, indent=2)
        self.logger.info("Отчёт сохранён: %s", self.settings.output_path)
    except OSError as exc:
        raise ConfigError(f"Не удалось сохранить отчёт: {exc}") from exc

Ingester._save = _save

## Задание 16. Логирование конвейера

In [22]:
def setup_logging():
    """Настраивает INFO для консоли и DEBUG для ingest.log, с ротацией 100 КБ и 3 архивами."""
    logger = logging.getLogger("ingester")
    logger.setLevel(logging.DEBUG); logger.handlers.clear(); logger.propagate = False
    console = logging.StreamHandler(); console.setLevel(logging.INFO)
    fh = RotatingFileHandler("ingest.log", maxBytes=100*1024, backupCount=3, encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(name)s %(message)s")
    console.setFormatter(fmt); fh.setFormatter(fmt)
    logger.addHandler(console); logger.addHandler(fh)
    return logger

pipeline_logger = setup_logging()

## Задание 17. CLI и коды завершения

In [25]:
def main() -> int:
    """Запускает импорт; возвращает 0 при успехе, 1 при ошибке импорта, 2 при ошибке настроек."""
    logger = setup_logging()
    try: cfg = Settings.from_env()
    except ConfigError as exc:
        logger.error("Конфигурация невалидна: %s", exc); return 2
    try:
        ingester = Ingester(cfg, logger)
        report = ingester.run()
        ingester._save(report)
    except IngestError as exc:
        logger.error("Импорт прерван: %s", exc); return 1
    logger.info("Готово: импортировано %d из %d", report["imported"], report["total_files"])
    return 0

# Планировщик может различать успешный запуск и разные типы сбоев по коду завершения.

## Задание 18. Проверка сценариев

In [26]:
exit_code = main()
report = json.loads(Path("catalog.json").read_text(encoding="utf-8"))
print("Код:", exit_code, "| найдено:", report["total_files"],
      "| импортировано:", report["imported"], "| пропущено:", report["skipped"])
print("Импортированные записи:", [r["name"] for r in report["records"]])
print("catalog.json:", Path("catalog.json").exists(), "| ingest.log:", Path("ingest.log").exists())
assert (report["total_files"], report["imported"], report["skipped"], exit_code) == (6, 3, 3, 0)

2026-10-09 13:12:56,260 WARNING ingester Документ пропущен: inbox\broken.json: Expecting value: line 1 column 19 (char 18)
2026-10-09 13:12:56,262 WARNING ingester Документ пропущен: inbox\no_keys.json: отсутствуют поля: id, name
2026-10-09 13:12:56,263 WARNING ingester Документ пропущен: inbox\not_object.json: JSON должен быть объектом (словарём)
2026-10-09 13:12:56,267 INFO ingester Отчёт сохранён: catalog.json
2026-10-09 13:12:56,269 INFO ingester Готово: импортировано 3 из 6


Код: 0 | найдено: 6 | импортировано: 3 | пропущено: 3
Импортированные записи: ['Alice', 'Bob', 'Carol']
catalog.json: True | ingest.log: True


## Задание 19. `.env` и `.gitignore`

In [27]:
Path(".env").write_text("SOURCE_DIR=inbox\nOUTPUT_PATH=catalog.json\nREQUIRED_KEYS=id,name\n", encoding="utf-8")
Path(".gitignore").write_text(
    ".env\nvenv/\n__pycache__/\n*.pyc\n*.log\n.idea/\n.vscode/\ninbox/\n.venv/\n.ipynb_checkpoints/\n",
    encoding="utf-8"
)
try:
    from dotenv import load_dotenv
    load_dotenv(".env", override=True)
    print("Настройки загружены через python-dotenv.")
except ImportError:
    for line in Path(".env").read_text(encoding="utf-8").splitlines():
        if "=" in line:
            key, value = line.split("=", 1); os.environ[key] = value
    print("python-dotenv отсутствует; переменные загружены резервным способом.")
print("Созданы .env и .gitignore. Не добавляйте .env в репозиторий.")

python-dotenv отсутствует; переменные загружены резервным способом.
Созданы .env и .gitignore. Не добавляйте .env в репозиторий.


## Задание 20. Единый pipeline

In [28]:
def generate_report(input_dir: str, output_path: str, required_keys: tuple) -> dict:
    """Создаёт настройки, запускает Ingester, сохраняет и возвращает отчёт."""
    cfg = Settings(input_dir, output_path, required_keys)
    if not cfg.source_dir.is_dir(): raise ConfigError(f"Нет каталога: {cfg.source_dir}")
    if not cfg.required_keys: raise ConfigError("Пустой список обязательных полей")
    logger = setup_logging()
    ingester = Ingester(cfg, logger)
    report = ingester.run()
    ingester._save(report)
    return report

def main_pipeline() -> int:
    """Оркестратор итогового конвейера: настройки, запуск и сводка."""
    logger = setup_logging()
    try:
        cfg = Settings.from_env()
        report = generate_report(str(cfg.source_dir), str(cfg.output_path), cfg.required_keys)
    except ConfigError as exc:
        logger.error("Ошибка конфигурации/сохранения: %s", exc); return 2
    except IngestError as exc:
        logger.error("Импорт прерван: %s", exc); return 1
    logger.info("Итог: %d импортировано, %d пропущено из %d",
                report["imported"], report["skipped"], report["total_files"])
    print("Отчёт:", cfg.output_path.resolve())
    print("Лог:", Path("ingest.log").resolve())
    return 0

final_code = main_pipeline()
print("Код завершения:", final_code)
assert final_code == 0

2026-10-09 13:13:14,846 WARNING ingester Документ пропущен: inbox\broken.json: Expecting value: line 1 column 19 (char 18)
2026-10-09 13:13:14,849 WARNING ingester Документ пропущен: inbox\no_keys.json: отсутствуют поля: id, name
2026-10-09 13:13:14,850 WARNING ingester Документ пропущен: inbox\not_object.json: JSON должен быть объектом (словарём)
2026-10-09 13:13:14,854 INFO ingester Отчёт сохранён: catalog.json
2026-10-09 13:13:14,855 INFO ingester Итог: 3 импортировано, 3 пропущено из 6


Отчёт: D:\maestria\I semestre\python-labs-gutierrez\catalog.json
Лог: D:\maestria\I semestre\python-labs-gutierrez\ingest.log
Код завершения: 0


In [1]:
import os, re, textwrap, json
from pathlib import Path
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH

outdir = Path("/mnt/data")
sql_path = outdir / "SEDERG_PAY_Modelo_Fisico_MySQL.sql"
docx_path = outdir / "SEDERG_PAY_Modelo_Fisico_MySQL.docx"

sql = r"""-- SEDERG_PAY - Modelo físico de datos
-- Motor: MySQL 8.0.36+ / MySQL 8.4 LTS
-- Storage engine: InnoDB | Charset: utf8mb4 | Collation: utf8mb4_0900_ai_ci
-- Zona horaria de aplicación: UTC. Montos financieros: DECIMAL, nunca FLOAT/DOUBLE.
-- IMPORTANTE: esquema base de desarrollo; validar legalmente y con proveedores antes de producción.
-- No almacenar PAN completo/CVV ni documentos de identidad en texto claro.
-- Las operaciones contables confirmadas son inmutables; corregir mediante asientos compensatorios.

CREATE DATABASE IF NOT EXISTS sederG_pay
  CHARACTER SET utf8mb4 COLLATE utf8mb4_0900_ai_ci;
USE sederG_pay;

SET NAMES utf8mb4;
SET time_zone = '+00:00';

CREATE TABLE countries (
  id SMALLINT UNSIGNED NOT NULL AUTO_INCREMENT,
  iso2 CHAR(2) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  iso3 CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  name_es VARCHAR(100) NOT NULL,
  name_local VARCHAR(100) NULL,
  is_active BOOLEAN NOT NULL DEFAULT TRUE,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  UNIQUE KEY uq_countries_iso2 (iso2),
  UNIQUE KEY uq_countries_iso3 (iso3)
) ENGINE=InnoDB;

CREATE TABLE currencies (
  code CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  name VARCHAR(80) NOT NULL,
  minor_units TINYINT UNSIGNED NOT NULL DEFAULT 2,
  is_active BOOLEAN NOT NULL DEFAULT TRUE,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (code),
  CONSTRAINT chk_currencies_minor_units CHECK (minor_units <= 6)
) ENGINE=InnoDB;

CREATE TABLE corridors (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  code VARCHAR(20) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  origin_country_id SMALLINT UNSIGNED NOT NULL,
  destination_country_id SMALLINT UNSIGNED NOT NULL,
  source_currency CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  destination_currency CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  status ENUM('DRAFT','UNDER_REVIEW','ACTIVE','SUSPENDED','CLOSED') NOT NULL DEFAULT 'DRAFT',
  legal_review_status ENUM('NOT_STARTED','IN_PROGRESS','APPROVED','REJECTED','EXPIRED') NOT NULL DEFAULT 'NOT_STARTED',
  legal_reviewed_at DATETIME(6) NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  updated_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6) ON UPDATE CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  UNIQUE KEY uq_corridors_code (code),
  KEY idx_corridors_route_status (origin_country_id,destination_country_id,status),
  CONSTRAINT fk_corridors_origin FOREIGN KEY (origin_country_id) REFERENCES countries(id),
  CONSTRAINT fk_corridors_destination FOREIGN KEY (destination_country_id) REFERENCES countries(id),
  CONSTRAINT fk_corridors_source_currency FOREIGN KEY (source_currency) REFERENCES currencies(code),
  CONSTRAINT fk_corridors_destination_currency FOREIGN KEY (destination_currency) REFERENCES currencies(code),
  CONSTRAINT chk_corridors_different_countries CHECK (origin_country_id <> destination_country_id)
) ENGINE=InnoDB;

CREATE TABLE users (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  public_id BINARY(16) NOT NULL,
  email VARCHAR(254) NULL,
  phone_e164 VARCHAR(20) NULL,
  password_hash VARCHAR(255) NOT NULL,
  status ENUM('PENDING','ACTIVE','LOCKED','SUSPENDED','CLOSED') NOT NULL DEFAULT 'PENDING',
  preferred_language VARCHAR(10) NOT NULL DEFAULT 'es',
  email_verified_at DATETIME(6) NULL,
  phone_verified_at DATETIME(6) NULL,
  last_login_at DATETIME(6) NULL,
  failed_login_count SMALLINT UNSIGNED NOT NULL DEFAULT 0,
  locked_until DATETIME(6) NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  updated_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6) ON UPDATE CURRENT_TIMESTAMP(6),
  deleted_at DATETIME(6) NULL,
  PRIMARY KEY (id),
  UNIQUE KEY uq_users_public_id (public_id),
  UNIQUE KEY uq_users_email (email),
  UNIQUE KEY uq_users_phone (phone_e164),
  KEY idx_users_status_created (status,created_at),
  CONSTRAINT chk_users_contact CHECK (email IS NOT NULL OR phone_e164 IS NOT NULL)
) ENGINE=InnoDB;

CREATE TABLE roles (
  id SMALLINT UNSIGNED NOT NULL AUTO_INCREMENT,
  code VARCHAR(50) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  name VARCHAR(100) NOT NULL,
  description VARCHAR(500) NULL,
  is_system BOOLEAN NOT NULL DEFAULT FALSE,
  PRIMARY KEY (id),
  UNIQUE KEY uq_roles_code (code)
) ENGINE=InnoDB;

CREATE TABLE permissions (
  id SMALLINT UNSIGNED NOT NULL AUTO_INCREMENT,
  code VARCHAR(100) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  description VARCHAR(255) NULL,
  PRIMARY KEY (id),
  UNIQUE KEY uq_permissions_code (code)
) ENGINE=InnoDB;

CREATE TABLE user_roles (
  user_id BIGINT UNSIGNED NOT NULL,
  role_id SMALLINT UNSIGNED NOT NULL,
  assigned_by_user_id BIGINT UNSIGNED NULL,
  assigned_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (user_id,role_id),
  KEY idx_user_roles_role (role_id,user_id),
  CONSTRAINT fk_user_roles_user FOREIGN KEY (user_id) REFERENCES users(id),
  CONSTRAINT fk_user_roles_role FOREIGN KEY (role_id) REFERENCES roles(id),
  CONSTRAINT fk_user_roles_assigner FOREIGN KEY (assigned_by_user_id) REFERENCES users(id)
) ENGINE=InnoDB;

CREATE TABLE role_permissions (
  role_id SMALLINT UNSIGNED NOT NULL,
  permission_id SMALLINT UNSIGNED NOT NULL,
  PRIMARY KEY (role_id,permission_id),
  CONSTRAINT fk_role_permissions_role FOREIGN KEY (role_id) REFERENCES roles(id),
  CONSTRAINT fk_role_permissions_permission FOREIGN KEY (permission_id) REFERENCES permissions(id)
) ENGINE=InnoDB;

CREATE TABLE customer_profiles (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  user_id BIGINT UNSIGNED NOT NULL,
  first_name VARCHAR(100) NOT NULL,
  middle_name VARCHAR(100) NULL,
  last_name VARCHAR(120) NOT NULL,
  date_of_birth DATE NULL,
  nationality_country_id SMALLINT UNSIGNED NULL,
  residence_country_id SMALLINT UNSIGNED NULL,
  address_line_encrypted VARBINARY(1024) NULL,
  city VARCHAR(100) NULL,
  postal_code VARCHAR(20) NULL,
  identity_status ENUM('NOT_STARTED','PENDING','VERIFIED','REJECTED','EXPIRED','REVIEW') NOT NULL DEFAULT 'NOT_STARTED',
  risk_level ENUM('LOW','MEDIUM','HIGH','PROHIBITED') NOT NULL DEFAULT 'MEDIUM',
  profile_version INT UNSIGNED NOT NULL DEFAULT 1,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  updated_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6) ON UPDATE CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  UNIQUE KEY uq_customer_profiles_user (user_id),
  KEY idx_customer_profiles_risk (risk_level,identity_status),
  CONSTRAINT fk_customer_profiles_user FOREIGN KEY (user_id) REFERENCES users(id),
  CONSTRAINT fk_customer_profiles_nationality FOREIGN KEY (nationality_country_id) REFERENCES countries(id),
  CONSTRAINT fk_customer_profiles_residence FOREIGN KEY (residence_country_id) REFERENCES countries(id)
) ENGINE=InnoDB;

CREATE TABLE identity_checks (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  customer_profile_id BIGINT UNSIGNED NOT NULL,
  provider_code VARCHAR(60) NOT NULL,
  provider_reference VARCHAR(120) NULL,
  check_type ENUM('IDENTITY','ADDRESS','LIVENESS','SANCTIONS','PEP','OTHER') NOT NULL,
  status ENUM('PENDING','PASSED','FAILED','MANUAL_REVIEW','EXPIRED','ERROR') NOT NULL,
  checked_at DATETIME(6) NULL,
  expires_at DATETIME(6) NULL,
  result_summary VARCHAR(1000) NULL,
  evidence_object_key VARCHAR(512) NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  UNIQUE KEY uq_identity_provider_ref (provider_code,provider_reference),
  KEY idx_identity_profile_status (customer_profile_id,status,created_at),
  KEY idx_identity_expiry (expires_at,status),
  CONSTRAINT fk_identity_checks_profile FOREIGN KEY (customer_profile_id) REFERENCES customer_profiles(id)
) ENGINE=InnoDB;

CREATE TABLE beneficiaries (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  owner_user_id BIGINT UNSIGNED NOT NULL,
  public_id BINARY(16) NOT NULL,
  first_name VARCHAR(100) NOT NULL,
  last_name VARCHAR(120) NOT NULL,
  relationship_code VARCHAR(40) NULL,
  country_id SMALLINT UNSIGNED NOT NULL,
  phone_e164 VARCHAR(20) NULL,
  email VARCHAR(254) NULL,
  address_encrypted VARBINARY(1024) NULL,
  status ENUM('ACTIVE','BLOCKED','ARCHIVED') NOT NULL DEFAULT 'ACTIVE',
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  updated_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6) ON UPDATE CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  UNIQUE KEY uq_beneficiaries_public_id (public_id),
  KEY idx_beneficiaries_owner_status (owner_user_id,status),
  KEY idx_beneficiaries_country (country_id),
  CONSTRAINT fk_beneficiaries_owner FOREIGN KEY (owner_user_id) REFERENCES users(id),
  CONSTRAINT fk_beneficiaries_country FOREIGN KEY (country_id) REFERENCES countries(id),
  CONSTRAINT chk_beneficiaries_contact CHECK (phone_e164 IS NOT NULL OR email IS NOT NULL)
) ENGINE=InnoDB;

CREATE TABLE payment_providers (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  code VARCHAR(60) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  name VARCHAR(150) NOT NULL,
  provider_type ENUM('BANK','CARD_PROCESSOR','CASH_NETWORK','WALLET','INTERNAL','OTHER') NOT NULL,
  country_id SMALLINT UNSIGNED NULL,
  status ENUM('DRAFT','TESTING','ACTIVE','SUSPENDED','TERMINATED') NOT NULL DEFAULT 'DRAFT',
  contract_reference VARCHAR(120) NULL,
  capabilities_json JSON NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  updated_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6) ON UPDATE CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  UNIQUE KEY uq_payment_providers_code (code),
  KEY idx_payment_providers_status_country (status,country_id),
  CONSTRAINT fk_payment_providers_country FOREIGN KEY (country_id) REFERENCES countries(id)
) ENGINE=InnoDB;

CREATE TABLE corridor_provider_methods (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  corridor_id BIGINT UNSIGNED NOT NULL,
  provider_id BIGINT UNSIGNED NOT NULL,
  side ENUM('COLLECTION','PAYOUT','BOTH') NOT NULL,
  method ENUM('BANK_ACCOUNT','BANK_CARD','CASH','WALLET','OTHER') NOT NULL,
  currency_code CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  min_amount DECIMAL(19,4) NULL,
  max_amount DECIMAL(19,4) NULL,
  status ENUM('DRAFT','ACTIVE','SUSPENDED','RETIRED') NOT NULL DEFAULT 'DRAFT',
  provider_config_json JSON NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  UNIQUE KEY uq_corridor_provider_method (corridor_id,provider_id,side,method,currency_code),
  KEY idx_cpm_status (status,corridor_id,side),
  CONSTRAINT fk_cpm_corridor FOREIGN KEY (corridor_id) REFERENCES corridors(id),
  CONSTRAINT fk_cpm_provider FOREIGN KEY (provider_id) REFERENCES payment_providers(id),
  CONSTRAINT fk_cpm_currency FOREIGN KEY (currency_code) REFERENCES currencies(code),
  CONSTRAINT chk_cpm_amounts CHECK (min_amount IS NULL OR max_amount IS NULL OR min_amount <= max_amount)
) ENGINE=InnoDB;

CREATE TABLE beneficiary_payout_methods (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  beneficiary_id BIGINT UNSIGNED NOT NULL,
  provider_method_id BIGINT UNSIGNED NOT NULL,
  label VARCHAR(80) NULL,
  account_holder_name VARCHAR(220) NULL,
  account_reference_encrypted VARBINARY(2048) NULL,
  account_reference_token VARCHAR(255) NULL,
  is_default BOOLEAN NOT NULL DEFAULT FALSE,
  status ENUM('ACTIVE','INVALID','BLOCKED','REMOVED') NOT NULL DEFAULT 'ACTIVE',
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  updated_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6) ON UPDATE CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  UNIQUE KEY uq_bpm_account_token (provider_method_id,account_reference_token),
  KEY idx_bpm_beneficiary_status (beneficiary_id,status),
  CONSTRAINT fk_bpm_beneficiary FOREIGN KEY (beneficiary_id) REFERENCES beneficiaries(id),
  CONSTRAINT fk_bpm_provider_method FOREIGN KEY (provider_method_id) REFERENCES corridor_provider_methods(id)
) ENGINE=InnoDB;

CREATE TABLE exchange_rate_sources (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  code VARCHAR(60) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  name VARCHAR(150) NOT NULL,
  source_type ENUM('PROVIDER','BANK','MARKET_DATA','MANUAL','OTHER') NOT NULL,
  status ENUM('ACTIVE','INACTIVE') NOT NULL DEFAULT 'ACTIVE',
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  UNIQUE KEY uq_exchange_rate_sources_code (code)
) ENGINE=InnoDB;

CREATE TABLE exchange_rate_snapshots (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  source_id BIGINT UNSIGNED NOT NULL,
  base_currency CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  quote_currency CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  rate DECIMAL(20,10) NOT NULL,
  observed_at DATETIME(6) NOT NULL,
  received_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  source_reference VARCHAR(150) NULL,
  is_approved BOOLEAN NOT NULL DEFAULT FALSE,
  PRIMARY KEY (id),
  UNIQUE KEY uq_rate_source_ref (source_id,source_reference),
  KEY idx_rate_pair_observed (base_currency,quote_currency,observed_at),
  KEY idx_rate_approval (is_approved,observed_at),
  CONSTRAINT fk_rate_source FOREIGN KEY (source_id) REFERENCES exchange_rate_sources(id),
  CONSTRAINT fk_rate_base_currency FOREIGN KEY (base_currency) REFERENCES currencies(code),
  CONSTRAINT fk_rate_quote_currency FOREIGN KEY (quote_currency) REFERENCES currencies(code),
  CONSTRAINT chk_rate_positive CHECK (rate > 0),
  CONSTRAINT chk_rate_distinct_currencies CHECK (base_currency <> quote_currency)
) ENGINE=InnoDB;

CREATE TABLE fee_rules (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  corridor_id BIGINT UNSIGNED NOT NULL,
  name VARCHAR(120) NOT NULL,
  fee_type ENUM('FIXED','PERCENT','FIXED_PLUS_PERCENT') NOT NULL,
  fixed_amount DECIMAL(19,4) NOT NULL DEFAULT 0,
  fixed_currency CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NULL,
  percentage_rate DECIMAL(9,6) NOT NULL DEFAULT 0,
  min_fee DECIMAL(19,4) NULL,
  max_fee DECIMAL(19,4) NULL,
  valid_from DATETIME(6) NOT NULL,
  valid_to DATETIME(6) NULL,
  status ENUM('DRAFT','ACTIVE','INACTIVE','EXPIRED') NOT NULL DEFAULT 'DRAFT',
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  KEY idx_fee_rules_corridor_status_dates (corridor_id,status,valid_from,valid_to),
  CONSTRAINT fk_fee_rules_corridor FOREIGN KEY (corridor_id) REFERENCES corridors(id),
  CONSTRAINT fk_fee_rules_currency FOREIGN KEY (fixed_currency) REFERENCES currencies(code),
  CONSTRAINT chk_fee_rules_nonnegative CHECK (fixed_amount >= 0 AND percentage_rate >= 0),
  CONSTRAINT chk_fee_rules_dates CHECK (valid_to IS NULL OR valid_to > valid_from),
  CONSTRAINT chk_fee_rules_minmax CHECK (min_fee IS NULL OR max_fee IS NULL OR min_fee <= max_fee)
) ENGINE=InnoDB;

CREATE TABLE quotes (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  public_id BINARY(16) NOT NULL,
  user_id BIGINT UNSIGNED NOT NULL,
  corridor_id BIGINT UNSIGNED NOT NULL,
  collection_method_id BIGINT UNSIGNED NOT NULL,
  payout_method_id BIGINT UNSIGNED NOT NULL,
  send_amount DECIMAL(19,4) NOT NULL,
  send_currency CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  receive_amount DECIMAL(19,4) NOT NULL,
  receive_currency CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  exchange_rate DECIMAL(20,10) NOT NULL,
  fee_amount DECIMAL(19,4) NOT NULL DEFAULT 0,
  fee_currency CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  total_debit_amount DECIMAL(19,4) NOT NULL,
  rate_snapshot_id BIGINT UNSIGNED NULL,
  fee_rule_id BIGINT UNSIGNED NULL,
  status ENUM('OPEN','ACCEPTED','EXPIRED','CANCELLED') NOT NULL DEFAULT 'OPEN',
  expires_at DATETIME(6) NOT NULL,
  accepted_at DATETIME(6) NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  UNIQUE KEY uq_quotes_public_id (public_id),
  KEY idx_quotes_user_status_created (user_id,status,created_at),
  KEY idx_quotes_expiry (status,expires_at),
  CONSTRAINT fk_quotes_user FOREIGN KEY (user_id) REFERENCES users(id),
  CONSTRAINT fk_quotes_corridor FOREIGN KEY (corridor_id) REFERENCES corridors(id),
  CONSTRAINT fk_quotes_collection_method FOREIGN KEY (collection_method_id) REFERENCES corridor_provider_methods(id),
  CONSTRAINT fk_quotes_payout_method FOREIGN KEY (payout_method_id) REFERENCES corridor_provider_methods(id),
  CONSTRAINT fk_quotes_send_currency FOREIGN KEY (send_currency) REFERENCES currencies(code),
  CONSTRAINT fk_quotes_receive_currency FOREIGN KEY (receive_currency) REFERENCES currencies(code),
  CONSTRAINT fk_quotes_fee_currency FOREIGN KEY (fee_currency) REFERENCES currencies(code),
  CONSTRAINT fk_quotes_rate_snapshot FOREIGN KEY (rate_snapshot_id) REFERENCES exchange_rate_snapshots(id),
  CONSTRAINT fk_quotes_fee_rule FOREIGN KEY (fee_rule_id) REFERENCES fee_rules(id),
  CONSTRAINT chk_quotes_amounts CHECK (send_amount > 0 AND receive_amount > 0 AND exchange_rate > 0 AND fee_amount >= 0 AND total_debit_amount > 0),
  CONSTRAINT chk_quotes_dates CHECK (expires_at > created_at)
) ENGINE=InnoDB;

CREATE TABLE transfer_requests (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  public_id BINARY(16) NOT NULL,
  public_reference VARCHAR(40) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  user_id BIGINT UNSIGNED NOT NULL,
  beneficiary_id BIGINT UNSIGNED NOT NULL,
  quote_id BIGINT UNSIGNED NOT NULL,
  corridor_id BIGINT UNSIGNED NOT NULL,
  collection_method_id BIGINT UNSIGNED NOT NULL,
  payout_method_id BIGINT UNSIGNED NOT NULL,
  send_amount DECIMAL(19,4) NOT NULL,
  send_currency CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  receive_amount DECIMAL(19,4) NOT NULL,
  receive_currency CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  exchange_rate DECIMAL(20,10) NOT NULL,
  fee_amount DECIMAL(19,4) NOT NULL,
  fee_currency CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  total_debit_amount DECIMAL(19,4) NOT NULL,
  status ENUM('CREATED','AWAITING_COMPLIANCE','AWAITING_FUNDS','FUNDS_RECEIVED','PROCESSING','PAYOUT_PENDING','COMPLETED','REJECTED','CANCELLED','FAILED','REFUND_PENDING','REFUNDED') NOT NULL DEFAULT 'CREATED',
  compliance_status ENUM('NOT_REQUIRED','PENDING','CLEAR','HOLD','REJECTED') NOT NULL DEFAULT 'PENDING',
  collection_status ENUM('NOT_STARTED','PENDING','CONFIRMED','FAILED','REVERSED') NOT NULL DEFAULT 'NOT_STARTED',
  payout_status ENUM('NOT_STARTED','PENDING','SENT','CONFIRMED','FAILED','REVERSED') NOT NULL DEFAULT 'NOT_STARTED',
  idempotency_key VARCHAR(128) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  updated_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6) ON UPDATE CURRENT_TIMESTAMP(6),
  completed_at DATETIME(6) NULL,
  cancelled_at DATETIME(6) NULL,
  row_version INT UNSIGNED NOT NULL DEFAULT 1,
  PRIMARY KEY (id),
  UNIQUE KEY uq_transfer_public_id (public_id),
  UNIQUE KEY uq_transfer_reference (public_reference),
  UNIQUE KEY uq_transfer_idempotency (user_id,idempotency_key),
  KEY idx_transfer_user_status_created (user_id,status,created_at),
  KEY idx_transfer_beneficiary_created (beneficiary_id,created_at),
  KEY idx_transfer_corridor_status (corridor_id,status,created_at),
  KEY idx_transfer_compliance (compliance_status,status,created_at),
  KEY idx_transfer_collection (collection_status,status),
  KEY idx_transfer_payout (payout_status,status),
  CONSTRAINT fk_transfer_user FOREIGN KEY (user_id) REFERENCES users(id),
  CONSTRAINT fk_transfer_beneficiary FOREIGN KEY (beneficiary_id) REFERENCES beneficiaries(id),
  CONSTRAINT fk_transfer_quote FOREIGN KEY (quote_id) REFERENCES quotes(id),
  CONSTRAINT fk_transfer_corridor FOREIGN KEY (corridor_id) REFERENCES corridors(id),
  CONSTRAINT fk_transfer_collection_method FOREIGN KEY (collection_method_id) REFERENCES corridor_provider_methods(id),
  CONSTRAINT fk_transfer_payout_method FOREIGN KEY (payout_method_id) REFERENCES corridor_provider_methods(id),
  CONSTRAINT fk_transfer_send_currency FOREIGN KEY (send_currency) REFERENCES currencies(code),
  CONSTRAINT fk_transfer_receive_currency FOREIGN KEY (receive_currency) REFERENCES currencies(code),
  CONSTRAINT fk_transfer_fee_currency FOREIGN KEY (fee_currency) REFERENCES currencies(code),
  CONSTRAINT chk_transfer_amounts CHECK (send_amount > 0 AND receive_amount > 0 AND exchange_rate > 0 AND fee_amount >= 0 AND total_debit_amount > 0)
) ENGINE=InnoDB;

CREATE TABLE transfer_state_events (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  transfer_id BIGINT UNSIGNED NOT NULL,
  from_status VARCHAR(40) NULL,
  to_status VARCHAR(40) NOT NULL,
  actor_user_id BIGINT UNSIGNED NULL,
  actor_type ENUM('CUSTOMER','OPERATOR','SYSTEM','PROVIDER','COMPLIANCE') NOT NULL,
  reason_code VARCHAR(80) NULL,
  notes VARCHAR(1000) NULL,
  metadata_json JSON NULL,
  occurred_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  KEY idx_tse_transfer_time (transfer_id,occurred_at),
  KEY idx_tse_status_time (to_status,occurred_at),
  CONSTRAINT fk_tse_transfer FOREIGN KEY (transfer_id) REFERENCES transfer_requests(id),
  CONSTRAINT fk_tse_actor FOREIGN KEY (actor_user_id) REFERENCES users(id)
) ENGINE=InnoDB;

CREATE TABLE compliance_checks (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  transfer_id BIGINT UNSIGNED NULL,
  user_id BIGINT UNSIGNED NOT NULL,
  check_type ENUM('SANCTIONS','PEP','ADVERSE_MEDIA','TRANSACTION_MONITORING','SOURCE_OF_FUNDS','LIMITS','OTHER') NOT NULL,
  provider_code VARCHAR(80) NULL,
  provider_reference VARCHAR(150) NULL,
  result ENUM('PENDING','CLEAR','MATCH','FALSE_POSITIVE','ESCALATED','ERROR') NOT NULL,
  risk_score DECIMAL(7,4) NULL,
  checked_at DATETIME(6) NULL,
  expires_at DATETIME(6) NULL,
  result_summary VARCHAR(1000) NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  UNIQUE KEY uq_compliance_provider_ref (provider_code,provider_reference),
  KEY idx_compliance_transfer_result (transfer_id,result,created_at),
  KEY idx_compliance_user_type (user_id,check_type,created_at),
  CONSTRAINT fk_compliance_transfer FOREIGN KEY (transfer_id) REFERENCES transfer_requests(id),
  CONSTRAINT fk_compliance_user FOREIGN KEY (user_id) REFERENCES users(id),
  CONSTRAINT chk_compliance_risk_score CHECK (risk_score IS NULL OR (risk_score >= 0 AND risk_score <= 100))
) ENGINE=InnoDB;

CREATE TABLE compliance_cases (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  case_reference VARCHAR(40) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  user_id BIGINT UNSIGNED NULL,
  transfer_id BIGINT UNSIGNED NULL,
  case_type ENUM('KYC','SANCTIONS','PEP','FRAUD','LIMITS','SOURCE_OF_FUNDS','OTHER') NOT NULL,
  priority ENUM('LOW','NORMAL','HIGH','CRITICAL') NOT NULL DEFAULT 'NORMAL',
  status ENUM('OPEN','IN_REVIEW','ESCALATED','APPROVED','REJECTED','CLOSED') NOT NULL DEFAULT 'OPEN',
  assigned_to_user_id BIGINT UNSIGNED NULL,
  resolution_code VARCHAR(80) NULL,
  resolution_notes VARCHAR(2000) NULL,
  opened_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  closed_at DATETIME(6) NULL,
  PRIMARY KEY (id),
  UNIQUE KEY uq_compliance_case_reference (case_reference),
  KEY idx_compliance_cases_status_priority (status,priority,opened_at),
  KEY idx_compliance_cases_user (user_id,opened_at),
  KEY idx_compliance_cases_transfer (transfer_id,opened_at),
  CONSTRAINT fk_cc_user FOREIGN KEY (user_id) REFERENCES users(id),
  CONSTRAINT fk_cc_transfer FOREIGN KEY (transfer_id) REFERENCES transfer_requests(id),
  CONSTRAINT fk_cc_assignee FOREIGN KEY (assigned_to_user_id) REFERENCES users(id),
  CONSTRAINT chk_compliance_case_dates CHECK (closed_at IS NULL OR closed_at >= opened_at)
) ENGINE=InnoDB;

CREATE TABLE compliance_case_events (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  case_id BIGINT UNSIGNED NOT NULL,
  actor_user_id BIGINT UNSIGNED NULL,
  event_type VARCHAR(60) NOT NULL,
  note VARCHAR(2000) NULL,
  metadata_json JSON NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  KEY idx_case_events_case_time (case_id,created_at),
  CONSTRAINT fk_case_events_case FOREIGN KEY (case_id) REFERENCES compliance_cases(id),
  CONSTRAINT fk_case_events_actor FOREIGN KEY (actor_user_id) REFERENCES users(id)
) ENGINE=InnoDB;

CREATE TABLE ledger_accounts (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  account_code VARCHAR(60) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  name VARCHAR(150) NOT NULL,
  account_type ENUM('ASSET','LIABILITY','EQUITY','INCOME','EXPENSE','CLEARING','MEMO') NOT NULL,
  currency_code CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  owner_user_id BIGINT UNSIGNED NULL,
  provider_id BIGINT UNSIGNED NULL,
  status ENUM('ACTIVE','FROZEN','CLOSED') NOT NULL DEFAULT 'ACTIVE',
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  UNIQUE KEY uq_ledger_account_code_currency (account_code,currency_code),
  KEY idx_ledger_accounts_owner (owner_user_id,currency_code,status),
  KEY idx_ledger_accounts_provider (provider_id,currency_code,status),
  CONSTRAINT fk_ledger_account_currency FOREIGN KEY (currency_code) REFERENCES currencies(code),
  CONSTRAINT fk_ledger_account_owner FOREIGN KEY (owner_user_id) REFERENCES users(id),
  CONSTRAINT fk_ledger_account_provider FOREIGN KEY (provider_id) REFERENCES payment_providers(id)
) ENGINE=InnoDB;

CREATE TABLE ledger_transactions (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  transaction_reference VARCHAR(50) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  transfer_id BIGINT UNSIGNED NULL,
  transaction_type ENUM('COLLECTION','PAYOUT','FEE','REFUND','RESERVATION','RELEASE','SETTLEMENT','ADJUSTMENT','REVERSAL','OTHER') NOT NULL,
  status ENUM('DRAFT','POSTED','REVERSED') NOT NULL DEFAULT 'DRAFT',
  idempotency_key VARCHAR(128) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  description VARCHAR(500) NULL,
  created_by_user_id BIGINT UNSIGNED NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  posted_at DATETIME(6) NULL,
  reverses_transaction_id BIGINT UNSIGNED NULL,
  PRIMARY KEY (id),
  UNIQUE KEY uq_ledger_tx_reference (transaction_reference),
  UNIQUE KEY uq_ledger_tx_idempotency (idempotency_key),
  KEY idx_ledger_tx_transfer (transfer_id,created_at),
  KEY idx_ledger_tx_status_posted (status,posted_at),
  CONSTRAINT fk_ledger_tx_transfer FOREIGN KEY (transfer_id) REFERENCES transfer_requests(id),
  CONSTRAINT fk_ledger_tx_creator FOREIGN KEY (created_by_user_id) REFERENCES users(id),
  CONSTRAINT fk_ledger_tx_reverses FOREIGN KEY (reverses_transaction_id) REFERENCES ledger_transactions(id)
) ENGINE=InnoDB;

CREATE TABLE ledger_entries (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  transaction_id BIGINT UNSIGNED NOT NULL,
  account_id BIGINT UNSIGNED NOT NULL,
  entry_side ENUM('DEBIT','CREDIT') NOT NULL,
  amount DECIMAL(19,4) NOT NULL,
  currency_code CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  memo VARCHAR(500) NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  KEY idx_ledger_entries_tx (transaction_id,id),
  KEY idx_ledger_entries_account_time (account_id,created_at),
  KEY idx_ledger_entries_currency_time (currency_code,created_at),
  CONSTRAINT fk_ledger_entries_tx FOREIGN KEY (transaction_id) REFERENCES ledger_transactions(id),
  CONSTRAINT fk_ledger_entries_account FOREIGN KEY (account_id) REFERENCES ledger_accounts(id),
  CONSTRAINT fk_ledger_entries_currency FOREIGN KEY (currency_code) REFERENCES currencies(code),
  CONSTRAINT chk_ledger_entries_amount CHECK (amount > 0)
) ENGINE=InnoDB;

CREATE TABLE local_accounts (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  provider_id BIGINT UNSIGNED NOT NULL,
  country_id SMALLINT UNSIGNED NOT NULL,
  currency_code CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  account_label VARCHAR(100) NOT NULL,
  account_reference_encrypted VARBINARY(2048) NULL,
  account_token VARCHAR(255) NULL,
  account_type ENUM('BANK','SETTLEMENT','CASH_FLOAT','WALLET','OTHER') NOT NULL,
  status ENUM('ACTIVE','FROZEN','CLOSED') NOT NULL DEFAULT 'ACTIVE',
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  updated_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6) ON UPDATE CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  UNIQUE KEY uq_local_account_token (provider_id,account_token),
  KEY idx_local_accounts_country_currency (country_id,currency_code,status),
  CONSTRAINT fk_local_accounts_provider FOREIGN KEY (provider_id) REFERENCES payment_providers(id),
  CONSTRAINT fk_local_accounts_country FOREIGN KEY (country_id) REFERENCES countries(id),
  CONSTRAINT fk_local_accounts_currency FOREIGN KEY (currency_code) REFERENCES currencies(code)
) ENGINE=InnoDB;

CREATE TABLE liquidity_snapshots (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  local_account_id BIGINT UNSIGNED NOT NULL,
  observed_at DATETIME(6) NOT NULL,
  confirmed_balance DECIMAL(19,4) NOT NULL,
  reserved_balance DECIMAL(19,4) NOT NULL DEFAULT 0,
  in_transit_balance DECIMAL(19,4) NOT NULL DEFAULT 0,
  unreconciled_balance DECIMAL(19,4) NOT NULL DEFAULT 0,
  source_reference VARCHAR(150) NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  KEY idx_liquidity_account_time (local_account_id,observed_at),
  CONSTRAINT fk_liquidity_local_account FOREIGN KEY (local_account_id) REFERENCES local_accounts(id),
  CONSTRAINT chk_liquidity_nonnegative CHECK (confirmed_balance >= 0 AND reserved_balance >= 0 AND in_transit_balance >= 0)
) ENGINE=InnoDB;

CREATE TABLE liquidity_reservations (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  transfer_id BIGINT UNSIGNED NOT NULL,
  local_account_id BIGINT UNSIGNED NOT NULL,
  amount DECIMAL(19,4) NOT NULL,
  currency_code CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  status ENUM('HELD','CONSUMED','RELEASED','EXPIRED') NOT NULL DEFAULT 'HELD',
  expires_at DATETIME(6) NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  resolved_at DATETIME(6) NULL,
  PRIMARY KEY (id),
  UNIQUE KEY uq_liquidity_reservation_transfer_account (transfer_id,local_account_id),
  KEY idx_liquidity_reservation_status_expiry (status,expires_at),
  CONSTRAINT fk_lr_transfer FOREIGN KEY (transfer_id) REFERENCES transfer_requests(id),
  CONSTRAINT fk_lr_local_account FOREIGN KEY (local_account_id) REFERENCES local_accounts(id),
  CONSTRAINT fk_lr_currency FOREIGN KEY (currency_code) REFERENCES currencies(code),
  CONSTRAINT chk_lr_amount CHECK (amount > 0)
) ENGINE=InnoDB;

CREATE TABLE settlement_batches (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  batch_reference VARCHAR(50) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  corridor_id BIGINT UNSIGNED NOT NULL,
  provider_id BIGINT UNSIGNED NOT NULL,
  currency_code CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  direction ENUM('PAYABLE','RECEIVABLE','NET') NOT NULL,
  status ENUM('DRAFT','CALCULATED','APPROVED','SENT','CONFIRMED','RECONCILED','FAILED','CANCELLED') NOT NULL DEFAULT 'DRAFT',
  total_amount DECIMAL(19,4) NOT NULL DEFAULT 0,
  item_count INT UNSIGNED NOT NULL DEFAULT 0,
  period_start DATETIME(6) NOT NULL,
  period_end DATETIME(6) NOT NULL,
  approved_by_user_id BIGINT UNSIGNED NULL,
  approved_at DATETIME(6) NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  UNIQUE KEY uq_settlement_batch_reference (batch_reference),
  KEY idx_settlement_status_period (status,period_start,period_end),
  CONSTRAINT fk_sb_corridor FOREIGN KEY (corridor_id) REFERENCES corridors(id),
  CONSTRAINT fk_sb_provider FOREIGN KEY (provider_id) REFERENCES payment_providers(id),
  CONSTRAINT fk_sb_currency FOREIGN KEY (currency_code) REFERENCES currencies(code),
  CONSTRAINT fk_sb_approver FOREIGN KEY (approved_by_user_id) REFERENCES users(id),
  CONSTRAINT chk_sb_period CHECK (period_end > period_start),
  CONSTRAINT chk_sb_total CHECK (total_amount >= 0)
) ENGINE=InnoDB;

CREATE TABLE settlement_items (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  settlement_batch_id BIGINT UNSIGNED NOT NULL,
  transfer_id BIGINT UNSIGNED NOT NULL,
  amount DECIMAL(19,4) NOT NULL,
  currency_code CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  item_status ENUM('INCLUDED','REMOVED','SETTLED','FAILED') NOT NULL DEFAULT 'INCLUDED',
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  UNIQUE KEY uq_settlement_item_batch_transfer (settlement_batch_id,transfer_id),
  KEY idx_settlement_items_transfer (transfer_id),
  CONSTRAINT fk_si_batch FOREIGN KEY (settlement_batch_id) REFERENCES settlement_batches(id),
  CONSTRAINT fk_si_transfer FOREIGN KEY (transfer_id) REFERENCES transfer_requests(id),
  CONSTRAINT fk_si_currency FOREIGN KEY (currency_code) REFERENCES currencies(code),
  CONSTRAINT chk_si_amount CHECK (amount > 0)
) ENGINE=InnoDB;

CREATE TABLE net_positions (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  corridor_id BIGINT UNSIGNED NOT NULL,
  provider_id BIGINT UNSIGNED NOT NULL,
  currency_code CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  period_start DATETIME(6) NOT NULL,
  period_end DATETIME(6) NOT NULL,
  gross_receivable DECIMAL(19,4) NOT NULL DEFAULT 0,
  gross_payable DECIMAL(19,4) NOT NULL DEFAULT 0,
  net_amount DECIMAL(19,4) NOT NULL DEFAULT 0,
  net_direction ENUM('RECEIVE','PAY','ZERO') NOT NULL DEFAULT 'ZERO',
  status ENUM('CALCULATED','REVIEWED','APPROVED','SETTLED','CANCELLED') NOT NULL DEFAULT 'CALCULATED',
  calculated_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  approved_by_user_id BIGINT UNSIGNED NULL,
  approved_at DATETIME(6) NULL,
  PRIMARY KEY (id),
  UNIQUE KEY uq_net_position_period (corridor_id,provider_id,currency_code,period_start,period_end),
  KEY idx_net_position_status_period (status,period_start),
  CONSTRAINT fk_np_corridor FOREIGN KEY (corridor_id) REFERENCES corridors(id),
  CONSTRAINT fk_np_provider FOREIGN KEY (provider_id) REFERENCES payment_providers(id),
  CONSTRAINT fk_np_currency FOREIGN KEY (currency_code) REFERENCES currencies(code),
  CONSTRAINT fk_np_approver FOREIGN KEY (approved_by_user_id) REFERENCES users(id),
  CONSTRAINT chk_np_period CHECK (period_end > period_start),
  CONSTRAINT chk_np_gross CHECK (gross_receivable >= 0 AND gross_payable >= 0)
) ENGINE=InnoDB;

CREATE TABLE payment_instructions (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  instruction_reference VARCHAR(50) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  transfer_id BIGINT UNSIGNED NOT NULL,
  provider_id BIGINT UNSIGNED NOT NULL,
  local_account_id BIGINT UNSIGNED NULL,
  instruction_type ENUM('COLLECTION','PAYOUT','REFUND','SETTLEMENT') NOT NULL,
  amount DECIMAL(19,4) NOT NULL,
  currency_code CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  status ENUM('CREATED','SUBMITTED','ACCEPTED','PENDING','SUCCEEDED','FAILED','CANCELLED','REVERSED') NOT NULL DEFAULT 'CREATED',
  provider_instruction_id VARCHAR(150) NULL,
  idempotency_key VARCHAR(128) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  submitted_at DATETIME(6) NULL,
  completed_at DATETIME(6) NULL,
  failure_code VARCHAR(100) NULL,
  failure_message VARCHAR(1000) NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  updated_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6) ON UPDATE CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  UNIQUE KEY uq_payment_instruction_reference (instruction_reference),
  UNIQUE KEY uq_payment_instruction_idempotency (provider_id,idempotency_key),
  UNIQUE KEY uq_payment_provider_instruction (provider_id,provider_instruction_id),
  KEY idx_payment_instruction_transfer_type (transfer_id,instruction_type,status),
  KEY idx_payment_instruction_status_created (status,created_at),
  CONSTRAINT fk_pi_transfer FOREIGN KEY (transfer_id) REFERENCES transfer_requests(id),
  CONSTRAINT fk_pi_provider FOREIGN KEY (provider_id) REFERENCES payment_providers(id),
  CONSTRAINT fk_pi_local_account FOREIGN KEY (local_account_id) REFERENCES local_accounts(id),
  CONSTRAINT fk_pi_currency FOREIGN KEY (currency_code) REFERENCES currencies(code),
  CONSTRAINT chk_pi_amount CHECK (amount > 0)
) ENGINE=InnoDB;

CREATE TABLE provider_events (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  provider_id BIGINT UNSIGNED NOT NULL,
  payment_instruction_id BIGINT UNSIGNED NULL,
  event_reference VARCHAR(180) NOT NULL,
  event_type VARCHAR(100) NOT NULL,
  signature_valid BOOLEAN NOT NULL DEFAULT FALSE,
  payload_json JSON NULL,
  received_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  processed_at DATETIME(6) NULL,
  processing_status ENUM('RECEIVED','PROCESSING','PROCESSED','IGNORED','FAILED') NOT NULL DEFAULT 'RECEIVED',
  processing_error VARCHAR(1000) NULL,
  PRIMARY KEY (id),
  UNIQUE KEY uq_provider_event_ref (provider_id,event_reference),
  KEY idx_provider_events_processing (processing_status,received_at),
  KEY idx_provider_events_instruction (payment_instruction_id,received_at),
  CONSTRAINT fk_provider_events_provider FOREIGN KEY (provider_id) REFERENCES payment_providers(id),
  CONSTRAINT fk_provider_events_instruction FOREIGN KEY (payment_instruction_id) REFERENCES payment_instructions(id)
) ENGINE=InnoDB;

CREATE TABLE reconciliation_batches (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  batch_reference VARCHAR(50) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  provider_id BIGINT UNSIGNED NOT NULL,
  local_account_id BIGINT UNSIGNED NULL,
  currency_code CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  source_file_key VARCHAR(512) NULL,
  period_start DATETIME(6) NOT NULL,
  period_end DATETIME(6) NOT NULL,
  status ENUM('IMPORTED','MATCHING','REVIEW_REQUIRED','RECONCILED','FAILED','CANCELLED') NOT NULL DEFAULT 'IMPORTED',
  imported_by_user_id BIGINT UNSIGNED NULL,
  imported_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  completed_at DATETIME(6) NULL,
  PRIMARY KEY (id),
  UNIQUE KEY uq_reconciliation_batch_reference (batch_reference),
  KEY idx_reconciliation_provider_status (provider_id,status,period_start),
  CONSTRAINT fk_rb_provider FOREIGN KEY (provider_id) REFERENCES payment_providers(id),
  CONSTRAINT fk_rb_local_account FOREIGN KEY (local_account_id) REFERENCES local_accounts(id),
  CONSTRAINT fk_rb_currency FOREIGN KEY (currency_code) REFERENCES currencies(code),
  CONSTRAINT fk_rb_importer FOREIGN KEY (imported_by_user_id) REFERENCES users(id),
  CONSTRAINT chk_rb_period CHECK (period_end > period_start)
) ENGINE=InnoDB;

CREATE TABLE reconciliation_records (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  batch_id BIGINT UNSIGNED NOT NULL,
  payment_instruction_id BIGINT UNSIGNED NULL,
  transfer_id BIGINT UNSIGNED NULL,
  provider_reference VARCHAR(180) NULL,
  external_amount DECIMAL(19,4) NULL,
  external_currency CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NULL,
  internal_amount DECIMAL(19,4) NULL,
  internal_currency CHAR(3) CHARACTER SET ascii COLLATE ascii_bin NULL,
  match_status ENUM('UNMATCHED','MATCHED','AMOUNT_MISMATCH','CURRENCY_MISMATCH','DUPLICATE','NEEDS_REVIEW','RESOLVED') NOT NULL DEFAULT 'UNMATCHED',
  difference_amount DECIMAL(19,4) NULL,
  notes VARCHAR(1000) NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  resolved_at DATETIME(6) NULL,
  PRIMARY KEY (id),
  KEY idx_reconciliation_records_batch_status (batch_id,match_status),
  KEY idx_reconciliation_records_provider_ref (provider_reference),
  KEY idx_reconciliation_records_transfer (transfer_id),
  CONSTRAINT fk_rr_batch FOREIGN KEY (batch_id) REFERENCES reconciliation_batches(id),
  CONSTRAINT fk_rr_instruction FOREIGN KEY (payment_instruction_id) REFERENCES payment_instructions(id),
  CONSTRAINT fk_rr_transfer FOREIGN KEY (transfer_id) REFERENCES transfer_requests(id),
  CONSTRAINT fk_rr_external_currency FOREIGN KEY (external_currency) REFERENCES currencies(code),
  CONSTRAINT fk_rr_internal_currency FOREIGN KEY (internal_currency) REFERENCES currencies(code)
) ENGINE=InnoDB;

CREATE TABLE reconciliation_resolutions (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  reconciliation_record_id BIGINT UNSIGNED NOT NULL,
  resolution_code VARCHAR(80) NOT NULL,
  resolution_notes VARCHAR(2000) NOT NULL,
  resolved_by_user_id BIGINT UNSIGNED NOT NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  KEY idx_reconciliation_resolution_record (reconciliation_record_id,created_at),
  CONSTRAINT fk_resolution_record FOREIGN KEY (reconciliation_record_id) REFERENCES reconciliation_records(id),
  CONSTRAINT fk_resolution_user FOREIGN KEY (resolved_by_user_id) REFERENCES users(id)
) ENGINE=InnoDB;

CREATE TABLE support_tickets (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  ticket_reference VARCHAR(40) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  user_id BIGINT UNSIGNED NULL,
  transfer_id BIGINT UNSIGNED NULL,
  category ENUM('TRANSFER','PAYMENT','ACCOUNT','KYC','TECHNICAL','OTHER') NOT NULL,
  priority ENUM('LOW','NORMAL','HIGH','URGENT') NOT NULL DEFAULT 'NORMAL',
  status ENUM('OPEN','ASSIGNED','WAITING_CUSTOMER','WAITING_PROVIDER','RESOLVED','CLOSED') NOT NULL DEFAULT 'OPEN',
  subject VARCHAR(200) NOT NULL,
  assigned_to_user_id BIGINT UNSIGNED NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  updated_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6) ON UPDATE CURRENT_TIMESTAMP(6),
  closed_at DATETIME(6) NULL,
  PRIMARY KEY (id),
  UNIQUE KEY uq_support_ticket_reference (ticket_reference),
  KEY idx_support_ticket_user_status (user_id,status,created_at),
  KEY idx_support_ticket_assignee_status (assigned_to_user_id,status,created_at),
  KEY idx_support_ticket_transfer (transfer_id),
  CONSTRAINT fk_support_ticket_user FOREIGN KEY (user_id) REFERENCES users(id),
  CONSTRAINT fk_support_ticket_transfer FOREIGN KEY (transfer_id) REFERENCES transfer_requests(id),
  CONSTRAINT fk_support_ticket_assignee FOREIGN KEY (assigned_to_user_id) REFERENCES users(id)
) ENGINE=InnoDB;

CREATE TABLE support_messages (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  ticket_id BIGINT UNSIGNED NOT NULL,
  sender_user_id BIGINT UNSIGNED NULL,
  sender_type ENUM('CUSTOMER','OPERATOR','SYSTEM') NOT NULL,
  message_text TEXT NOT NULL,
  attachment_object_key VARCHAR(512) NULL,
  is_internal BOOLEAN NOT NULL DEFAULT FALSE,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  KEY idx_support_messages_ticket_time (ticket_id,created_at),
  CONSTRAINT fk_support_messages_ticket FOREIGN KEY (ticket_id) REFERENCES support_tickets(id),
  CONSTRAINT fk_support_messages_sender FOREIGN KEY (sender_user_id) REFERENCES users(id)
) ENGINE=InnoDB;

CREATE TABLE notifications (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  user_id BIGINT UNSIGNED NOT NULL,
  transfer_id BIGINT UNSIGNED NULL,
  channel ENUM('EMAIL','SMS','PUSH','IN_APP') NOT NULL,
  template_code VARCHAR(100) NOT NULL,
  destination_masked VARCHAR(180) NULL,
  status ENUM('QUEUED','SENT','DELIVERED','FAILED','CANCELLED') NOT NULL DEFAULT 'QUEUED',
  provider_reference VARCHAR(180) NULL,
  attempts TINYINT UNSIGNED NOT NULL DEFAULT 0,
  scheduled_at DATETIME(6) NULL,
  sent_at DATETIME(6) NULL,
  delivered_at DATETIME(6) NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  KEY idx_notifications_user_status (user_id,status,created_at),
  KEY idx_notifications_queue (status,scheduled_at),
  KEY idx_notifications_transfer (transfer_id),
  CONSTRAINT fk_notifications_user FOREIGN KEY (user_id) REFERENCES users(id),
  CONSTRAINT fk_notifications_transfer FOREIGN KEY (transfer_id) REFERENCES transfer_requests(id)
) ENGINE=InnoDB;

CREATE TABLE audit_events (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  actor_user_id BIGINT UNSIGNED NULL,
  actor_type ENUM('CUSTOMER','OPERATOR','SYSTEM','PROVIDER','COMPLIANCE') NOT NULL,
  action_code VARCHAR(100) NOT NULL,
  entity_type VARCHAR(80) NOT NULL,
  entity_id VARCHAR(100) NULL,
  correlation_id VARCHAR(100) NULL,
  ip_address VARBINARY(16) NULL,
  user_agent VARCHAR(512) NULL,
  before_json JSON NULL,
  after_json JSON NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  KEY idx_audit_actor_time (actor_user_id,created_at),
  KEY idx_audit_entity (entity_type,entity_id,created_at),
  KEY idx_audit_action_time (action_code,created_at),
  KEY idx_audit_correlation (correlation_id),
  CONSTRAINT fk_audit_actor FOREIGN KEY (actor_user_id) REFERENCES users(id)
) ENGINE=InnoDB;

CREATE TABLE idempotency_records (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  scope_code VARCHAR(80) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  idempotency_key VARCHAR(128) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  actor_user_id BIGINT UNSIGNED NULL,
  request_hash BINARY(32) NOT NULL,
  response_status SMALLINT UNSIGNED NULL,
  response_json JSON NULL,
  resource_type VARCHAR(80) NULL,
  resource_id VARCHAR(100) NULL,
  status ENUM('IN_PROGRESS','COMPLETED','FAILED') NOT NULL DEFAULT 'IN_PROGRESS',
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  expires_at DATETIME(6) NULL,
  completed_at DATETIME(6) NULL,
  PRIMARY KEY (id),
  UNIQUE KEY uq_idempotency_scope_key (scope_code,idempotency_key),
  KEY idx_idempotency_actor_created (actor_user_id,created_at),
  KEY idx_idempotency_expiry (expires_at,status),
  CONSTRAINT fk_idempotency_actor FOREIGN KEY (actor_user_id) REFERENCES users(id)
) ENGINE=InnoDB;

CREATE TABLE configuration_versions (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  config_key VARCHAR(120) CHARACTER SET ascii COLLATE ascii_bin NOT NULL,
  version_number INT UNSIGNED NOT NULL,
  config_json JSON NOT NULL,
  status ENUM('DRAFT','ACTIVE','RETIRED') NOT NULL DEFAULT 'DRAFT',
  effective_from DATETIME(6) NOT NULL,
  effective_to DATETIME(6) NULL,
  created_by_user_id BIGINT UNSIGNED NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  UNIQUE KEY uq_config_key_version (config_key,version_number),
  KEY idx_config_active_dates (config_key,status,effective_from,effective_to),
  CONSTRAINT fk_config_creator FOREIGN KEY (created_by_user_id) REFERENCES users(id),
  CONSTRAINT chk_config_dates CHECK (effective_to IS NULL OR effective_to > effective_from)
) ENGINE=InnoDB;

CREATE TABLE secure_attachments (
  id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  owner_user_id BIGINT UNSIGNED NULL,
  entity_type VARCHAR(80) NOT NULL,
  entity_id VARCHAR(100) NOT NULL,
  object_key VARCHAR(512) NOT NULL,
  content_type VARCHAR(120) NOT NULL,
  file_size_bytes BIGINT UNSIGNED NOT NULL,
  sha256_hash BINARY(32) NOT NULL,
  encryption_key_reference VARCHAR(255) NOT NULL,
  malware_scan_status ENUM('PENDING','CLEAN','INFECTED','ERROR') NOT NULL DEFAULT 'PENDING',
  retention_until DATETIME(6) NULL,
  created_at DATETIME(6) NOT NULL DEFAULT CURRENT_TIMESTAMP(6),
  PRIMARY KEY (id),
  UNIQUE KEY uq_secure_attachment_object_key (object_key),
  KEY idx_secure_attachment_entity (entity_type,entity_id),
  KEY idx_secure_attachment_owner (owner_user_id,created_at),
  KEY idx_secure_attachment_retention (retention_until),
  CONSTRAINT fk_secure_attachment_owner FOREIGN KEY (owner_user_id) REFERENCES users(id)
) ENGINE=InnoDB;

-- Datos de referencia mínimos. Confirmar con asesoría y proveedores antes de activar corredores.
INSERT INTO countries (iso2, iso3, name_es, name_local) VALUES
('RU','RUS','Rusia','Россия'),
('CU','CUB','Cuba','Cuba')
ON DUPLICATE KEY UPDATE name_es=VALUES(name_es), name_local=VALUES(name_local);

INSERT INTO currencies (code, name, minor_units) VALUES
('RUB','Rublo ruso',2),
('CUP','Peso cubano',2),
('USD','Dólar estadounidense',2),
('EUR','Euro',2)
ON DUPLICATE KEY UPDATE name=VALUES(name), minor_units=VALUES(minor_units);

-- Validaciones obligatorias en la capa de servicio / transacción:
-- 1) Verificar que colección y pago pertenezcan al corredor y a la dirección correcta.
-- 2) Verificar balance contable por moneda en cada ledger_transaction antes de POSTED.
-- 3) No modificar ni borrar ledger_entries de transacciones POSTED; revertir con transacción compensatoria.
-- 4) Reservar liquidez con transacción InnoDB + SELECT ... FOR UPDATE o UPDATE condicional atómico.
-- 5) Las CHECK no validan sumas entre filas; el equilibrio débito/crédito debe verificarse en servicio.
-- 6) Validar ownership del beneficiario, vigencia de quote, KYC/AML, límites y proveedor activo.
-- 7) El saldo contable no equivale al saldo bancario confirmado; conciliar mediante reconciliation_records.
-- 8) No activar transferencias reales hasta completar revisión legal, contratos y permisos en ambos países.
"""

sql_path.write_text(sql, encoding="utf-8")

# Create document and parse CREATE TABLE blocks into a field-level data dictionary.
doc = Document()
sec = doc.sections[0]
sec.top_margin = Inches(0.65)
sec.bottom_margin = Inches(0.65)
sec.left_margin = Inches(0.7)
sec.right_margin = Inches(0.7)

styles = doc.styles
styles['Normal'].font.name = 'Arial'
styles['Normal'].font.size = Pt(9)
styles['Title'].font.name = 'Arial'
styles['Title'].font.size = Pt(22)
for sty in ['Heading 1','Heading 2','Heading 3']:
    styles[sty].font.name = 'Arial'

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
r = p.add_run("SEDERG_PAY\nModelo físico de datos — MySQL")
r.bold = True
r.font.size = Pt(22)
p2 = doc.add_paragraph()
p2.alignment = WD_ALIGN_PARAGRAPH.CENTER
p2.add_run("Versión 1.0 | 10 de octubre de 2026 | Documento técnico de diseño").italic = True

doc.add_heading("1. Objetivo y alcance", level=1)
doc.add_paragraph(
    "Este documento define el modelo físico inicial de SEDERG_PAY para MySQL: tablas, campos, tipos, "
    "claves primarias, claves foráneas, índices y restricciones. Cubre identidad, beneficiarios, "
    "cotizaciones, transferencias, cumplimiento, ledger contable, liquidez, compensación, proveedores, "
    "conciliación, soporte, notificaciones, auditoría e idempotencia."
)
doc.add_paragraph(
    "El esquema es una base técnica de desarrollo y no constituye autorización para operar remesas. "
    "La disponibilidad de métodos de cobro/pago, la custodia de fondos, el tratamiento de datos y las "
    "obligaciones AML/KYC deben validarse por separado en Rusia y Cuba con asesores y proveedores autorizados."
)

doc.add_heading("2. Plataforma y convenciones", level=1)
for item in [
    "Motor: MySQL 8.0.36 o posterior; preferible evaluar MySQL 8.4 LTS para despliegues nuevos. Confirmar soporte de CHECK y collation en la versión elegida.",
    "Motor de almacenamiento: InnoDB. Codificación: utf8mb4. Collation del esquema: utf8mb4_0900_ai_ci.",
    "Fechas: DATETIME(6), guardadas en UTC. La aplicación configura UTC y convierte a la zona horaria del usuario solo para presentación.",
    "Importes: DECIMAL(19,4); tasas: DECIMAL(20,10). Nunca usar FLOAT o DOUBLE para dinero.",
    "Identificadores internos: BIGINT UNSIGNED AUTO_INCREMENT; identificadores públicos: BINARY(16) para UUID binario.",
    "Datos sensibles: campos cifrados o tokenizados. No guardar PAN completo, CVV, contraseñas en claro ni documentos de identidad en texto claro.",
    "JSON se reserva para metadatos variables; importes, monedas, estados y referencias críticas usan columnas tipadas.",
    "Las tablas con historial/auditoría deben tener retención, acceso y monitorización definidos."
]:
    doc.add_paragraph(item, style='List Bullet')

doc.add_heading("3. Convenciones de integridad", level=1)
for item in [
    "Todas las relaciones principales usan claves foráneas explícitas. No se usa ON DELETE CASCADE para registros financieros, de cumplimiento o auditoría.",
    "Los importes de ledger_entries deben ser positivos y cada transacción POSTED debe cuadrar por moneda. MySQL no puede imponer mediante CHECK una suma entre varias filas; el servicio transaccional debe validarla.",
    "Los asientos POSTED son inmutables. Las correcciones se realizan mediante transacciones compensatorias vinculadas por reverses_transaction_id.",
    "La idempotencia se aplica en operaciones de transferencia, instrucciones de pago y endpoints mutables. Repetir una clave no debe duplicar una operación.",
    "Las reservas de liquidez se ejecutan en transacciones InnoDB con SELECT ... FOR UPDATE o actualización condicional atómica, con reintentos controlados para deadlocks.",
    "Los índices únicos sobre campos opcionales permiten múltiples NULL en MySQL; la aplicación debe decidir si una referencia del proveedor puede estar ausente."
]:
    doc.add_paragraph(item, style='List Bullet')

# Parse tables
blocks = re.findall(r"CREATE TABLE (\w+) \((.*?)\) ENGINE=InnoDB;", sql, flags=re.S)
doc.add_heading("4. Diccionario de datos físico", level=1)
doc.add_paragraph(f"El DDL incluye {len(blocks)} tablas. La tabla siguiente resume cada columna y marca PK, FK, UQ y los principales índices.")
for table_name, body in blocks:
    doc.add_heading(table_name, level=2)
    # get table-level constraints separately
    lines = [ln.strip().rstrip(',') for ln in body.splitlines() if ln.strip()]
    rows = []
    constraints = []
    for ln in lines:
        if ln.startswith(("PRIMARY KEY","UNIQUE KEY","KEY ","CONSTRAINT ")):
            constraints.append(ln)
            continue
        # column line: name then type and rest
        m = re.match(r"`?([A-Za-z_][A-Za-z0-9_]*)`?\s+(.+)", ln)
        if m and not ln.startswith(("CHECK","FOREIGN KEY")):
            col, rest = m.group(1), m.group(2)
            # cut inline definitions into compact detail
            flags = []
            if re.search(r"\bAUTO_INCREMENT\b", rest): flags.append("AUTO_INCREMENT")
            if re.search(r"\bNOT NULL\b", rest): flags.append("NOT NULL")
            if re.search(r"\bPRIMARY KEY\b", rest): flags.append("PK")
            # Identify FKs from table constraints below
            typ = rest.split(" NOT NULL")[0].split(" NULL")[0].split(" DEFAULT ")[0].split(" AUTO_INCREMENT")[0]
            rows.append((col, typ[:80], ", ".join(flags)))
    # add FK markers to fields based on constraints
    for i, (col, typ, flags) in enumerate(rows):
        if any(re.search(r"FOREIGN KEY \(" + re.escape(col) + r"\)", c) for c in constraints):
            flags = (flags + ", " if flags else "") + "FK"
        if any(re.search(r"UNIQUE KEY \w+ \([^)]*\b" + re.escape(col) + r"\b", c) for c in constraints):
            flags = (flags + ", " if flags else "") + "UQ/índice"
        rows[i] = (col, typ, flags)
    table = doc.add_table(rows=1, cols=3)
    table.style = 'Light Shading Accent 1'
    hdr = table.rows[0].cells
    hdr[0].text, hdr[1].text, hdr[2].text = "Campo", "Tipo / definición", "Atributos"
    for col, typ, flags in rows:
        cells = table.add_row().cells
        cells[0].text = col
        cells[1].text = typ
        cells[2].text = flags
    if constraints:
        para = doc.add_paragraph()
        para.add_run("Claves e índices: ").bold = True
        para.add_run("; ".join(constraints))
    # show checks / foreign keys in compact text, no overly long paragraphs
    check_lines = [c for c in constraints if c.startswith("CONSTRAINT")]
    if check_lines:
        para = doc.add_paragraph()
        para.add_run("Restricciones: ").bold = True
        para.add_run(" | ".join(check_lines))

doc.add_heading("5. Reglas que debe implementar la aplicación", level=1)
rules = [
    "Verificar que collection_method_id y payout_method_id estén activos, pertenezcan al corredor y tengan el lado operativo correcto.",
    "Validar que el beneficiario pertenece al usuario iniciador y que el método de pago seleccionado pertenece al beneficiario.",
    "Revalidar la cotización al crear la transferencia: propiedad, estado OPEN, expiración, moneda, importes y métodos.",
    "Comprobar estado de KYC/AML, sanciones, límites por operación/periodo y casos abiertos antes de autorizar el movimiento.",
    "Al registrar asientos: bloquear los registros necesarios, validar que débitos y créditos coincidan por moneda y marcar POSTED dentro de la misma transacción.",
    "Separar el estado de transferencia del estado de cobro, pago, cumplimiento y contabilidad; no inferir pago confirmado solo por una solicitud enviada.",
    "Verificar firma y autenticidad de eventos entrantes de proveedores, almacenar referencia externa única y procesar webhooks idempotentemente.",
    "No permitir que el cálculo de compensación bilateral o una posición neta se interprete como liquidez disponible; debe existir saldo confirmado y capacidad real de liquidación.",
    "Auditar cambios de configuración, roles, límites, estados, reservas y operaciones administrativas."
]
for item in rules:
    doc.add_paragraph(item, style='List Number')

doc.add_heading("6. Índices y restricciones críticas", level=1)
doc.add_paragraph(
    "El DDL define claves únicas para identificadores públicos, referencias de transferencia, claves de idempotencia, "
    "referencias de proveedor y versiones de configuración; índices compuestos para búsquedas por estado/fecha; "
    "y restricciones CHECK para importes positivos, rangos y periodos. Los índices adicionales deben justificarse "
    "con consultas reales y planes EXPLAIN, evitando indexar indiscriminadamente campos de alta cardinalidad."
)
doc.add_paragraph(
    "Antes de producción deben revisarse límites máximos, retención legal, borrado lógico, enmascaramiento de logs, "
    "gestión de claves de cifrado, copias de seguridad, recuperación ante desastres y segregación de funciones."
)

doc.add_heading("7. Notas de implementación MySQL", level=1)
for item in [
    "Ejecutar el archivo SQL en una instancia de prueba antes de integrarlo con Yii2. El nombre de base de datos del DDL es sederG_pay.",
    "En despliegues con MySQL anterior a 8.0.16, no confiar en CHECK: actualizar a una versión soportada.",
    "Revisar que el charset/collation esté disponible en la versión concreta de MySQL/MariaDB; este DDL apunta a MySQL 8.x, no a MariaDB.",
    "Definir migraciones Yii2 versionadas en lugar de aplicar cambios manuales en producción.",
    "Los campos JSON pueden necesitar validación de esquema en la aplicación; no deben ser la única fuente de datos críticos.",
    "No ejecutar transferencias reales con cuentas personales. Utilizar entidades, contratos y proveedores aprobados y confirmar la viabilidad regulatoria del corredor Rusia–Cuba."
]:
    doc.add_paragraph(item, style='List Bullet')

doc.add_heading("8. Entregables", level=1)
doc.add_paragraph("El archivo SEDERG_PAY_Modelo_Fisico_MySQL.sql contiene el DDL ejecutable, las claves, índices, restricciones y comentarios de implementación.")
doc.save(docx_path)

print(f"Archivos creados:\n- {sql_path}\n- {docx_path}\nTablas incluidas: {len(blocks)}")


FileNotFoundError: [Errno 2] No such file or directory: '\\mnt\\data\\SEDERG_PAY_Modelo_Fisico_MySQL.sql'